<a href="https://colab.research.google.com/github/will-mccormack/CS-M148-Proj/blob/main/Project_4_Check_In.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%pip install scikit-lego

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.3/227.3 kB 12.8 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import scipy as sp
import matplotlib as mpl
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import time
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor


In [4]:
cleaned_data = pd.read_csv("https://raw.githubusercontent.com/will-mccormack/CS-M148-Proj/main/Data/train.csv")
validation_data = pd.read_csv("https://raw.githubusercontent.com/will-mccormack/CS-M148-Proj/main/Data/validation.csv")

In [5]:
#cleaned_data.columns
validation_data.columns

Index(['popularity', 'duration_ms', 'explicit', 'danceability', 'energy',
       'key', 'loudness', 'mode', 'speechiness', 'acousticness',
       'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature',
       'track_genre'],
      dtype='object')

In [6]:
# drop features we've determined in previous weeks
train_data = cleaned_data.drop(["key","acousticness"], axis=1)
test_data = validation_data

In [21]:
from pandas.api.types import is_string_dtype
# find knn for one row
def find_knn(k, train_data, new_point, class_col, p):
  # create empty array for minkowski distance
  distance_calc = pd.Series(np.zeros(len(train_data)), index=train_data.index)
  # go through each numeric column and calculate the distance from the test data row
  for col_name in train_data.columns:
    if (col_name != class_col) and (not is_string_dtype(train_data[col_name])):
      distance_calc += abs(train_data[col_name]-new_point[col_name])**p
    else:
      continue
  distance_calc = distance_calc**(1/p) # finish distance calc
  smallest_k = distance_calc.nsmallest(k) # find smallest distance
  smallest_k_rows = smallest_k.index # find rows that correspond
  votes = train_data.loc[smallest_k_rows,class_col] # find classifiers for smallest k
  voted_class = votes.mode()[0] # get most voted class
  return voted_class

In [25]:
from sklearn.preprocessing import StandardScaler
# find knn for entire dataset
def knn_predict(k, train_data, test_data, class_col, p):
  # find numeric columns for analysis
  numeric_cols = []
  for col_name in train_data.columns:
    if (col_name != class_col) and (not is_string_dtype(train_data[col_name])):
      numeric_cols.append(col_name)
  # scale columns so predictors on a larger scale do not dominate
  scaler = StandardScaler()
  scaler.fit(train_data[numeric_cols])
  train_scaled_np = scaler.transform(train_data[numeric_cols])
  test_scaled_np = scaler.transform(test_data[numeric_cols])
  # rearrange data into a dataframe from np.array
  train_data_scaled = train_data.copy()
  test_data_scaled = test_data.copy()
  train_data_scaled[numeric_cols] = train_scaled_np
  test_data_scaled[numeric_cols] = test_scaled_np
  # find knn vote for each row in dataset
  predictions = test_data_scaled.apply(
      lambda new_point: find_knn(k, train_data_scaled, new_point, class_col, p),
      axis=1
  )
  return predictions

In [23]:
k = 3
p= 2

predictions = knn_predict(k, train_data, test_data, 'explicit', p)

predictions.head(5)

,0
0,False
1,False
2,False
3,False
4,True


In [24]:
truth_table = pd.DataFrame({
    'truth': test_data['explicit'],
    'prediction': predictions
})

truth_table.head(5)

,truth,prediction
0,False,False
1,False,False
2,False,False
3,False,False
4,False,True
